# 00 - Load and Clean
**QSS 20 Final Project | Olivia Tak**

This notebook loads the raw YouTube trending dataset and inspects its structure.

## Imports

In [1]:
import pandas as pd
import numpy as np
import json

## Load raw data

In [2]:
df = pd.read_csv('../data/US_youtube_trending_data.csv')
print(df.shape)
print(df.columns.tolist())

(268787, 16)
['video_id', 'title', 'publishedAt', 'channelId', 'channelTitle', 'categoryId', 'trending_date', 'tags', 'view_count', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'description']


## Initial inspection

In [3]:
df.head()

,video_id,title,publishedAt,channelId,channelTitle,categoryId,trending_date,tags,view_count,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,description
0,3C66w5Z0ixs,I ASKED HER TO BE MY GIRLFRIEND...,2020-08-11T19:20:14Z,UCvtRTOMP2TqYqu51xNrqAzg,Brawadis,22,2020-08-12T00:00:00Z,brawadis|prank|basketball|skits|ghost|funny vi...,1514614,156908,5855,35313,https://i.ytimg.com/vi/3C66w5Z0ixs/default.jpg,False,False,SUBSCRIBE to BRAWADIS ▶ http://bit.ly/Subscrib...
1,M9Pmf9AB4Mo,Apex Legends | Stories from the Outlands – “Th...,2020-08-11T17:00:10Z,UC0ZV6M2THA81QT9hrVWJG3A,Apex Legends,20,2020-08-12T00:00:00Z,Apex Legends|Apex Legends characters|new Apex ...,2381688,146739,2794,16549,https://i.ytimg.com/vi/M9Pmf9AB4Mo/default.jpg,False,False,"While running her own modding shop, Ramya Pare..."
2,J78aPJ3VyNs,I left youtube for a month and THIS is what ha...,2020-08-11T16:34:06Z,UCYzPXprvl5Y-Sf0g4vX-m6g,jacksepticeye,24,2020-08-12T00:00:00Z,jacksepticeye|funny|funny meme|memes|jacksepti...,2038853,353787,2628,40221,https://i.ytimg.com/vi/J78aPJ3VyNs/default.jpg,False,False,I left youtube for a month and this is what ha...
3,kXLn3HkpjaA,XXL 2020 Freshman Class Revealed - Official An...,2020-08-11T16:38:55Z,UCbg_UMjlHJg_19SZckaKajg,XXL,10,2020-08-12T00:00:00Z,xxl freshman|xxl freshmen|2020 xxl freshman|20...,496771,23251,1856,7647,https://i.ytimg.com/vi/kXLn3HkpjaA/default.jpg,False,False,Subscribe to XXL → http://bit.ly/subscribe-xxl...
4,VIUo6yapDbc,Ultimate DIY Home Movie Theater for The LaBran...,2020-08-11T15:10:05Z,UCDVPcEbVLQgLZX0Rt6jo34A,Mr. Kate,26,2020-08-12T00:00:00Z,The LaBrant Family|DIY|Interior Design|Makeove...,1123889,45802,964,2196,https://i.ytimg.com/vi/VIUo6yapDbc/default.jpg,False,False,Transforming The LaBrant Family's empty white ...


In [4]:
df.isnull().sum()

video_id                0
title                   0
publishedAt             0
channelId               0
channelTitle            0
categoryId              0
trending_date           0
tags                    0
view_count              0
likes                   0
dislikes                0
comment_count           0
thumbnail_link          0
comments_disabled       0
ratings_disabled        0
description          4549
dtype: int64

In [5]:
# check date range of the full dataset
df['trending_date'] = pd.to_datetime(df['trending_date'])
print(df['trending_date'].min())
print(df['trending_date'].max())

2020-08-12 00:00:00+00:00
2024-04-15 00:00:00+00:00


## Filter to 2023

In [6]:
df['trending_date'] = pd.to_datetime(df['trending_date']) #converts it to datetime object

df_2023 = df[df['trending_date'].dt.year == 2023].copy()

print(f"Full dataset: {len(df)} rows")
print(f"2023 only: {len(df_2023)} rows")

Full dataset: 268787 rows
2023 only: 72397 rows


## Making a New 'Days Trending' Variable

In [7]:
# count days each video appeared on trending
days_trending = df_2023.groupby('video_id')['trending_date'].count().reset_index()
days_trending.columns = ['video_id', 'days_trending']
print(days_trending['days_trending'].describe())

count    12176.000000
mean         5.945877
std          1.869705
min          1.000000
25%          5.000000
50%          6.000000
75%          7.000000
max         37.000000
Name: days_trending, dtype: float64


In [8]:
# deduplicate keeping first trending appearance, then merge count back in
df_2023 = df_2023.sort_values('trending_date')
df_2023 = df_2023.drop_duplicates(subset='video_id', keep='first')
df_2023 = df_2023.merge(days_trending, on='video_id', how='left')

print(f"After deduplication: {len(df_2023)} rows")
print(df_2023[['title', 'days_trending']].head(10))

After deduplication: 12176 rows
                                               title  days_trending
0  Peach Bowl: Ohio State Buckeyes vs. Georgia Bu...              9
1                              Tom MacDonald - Ghost              4
2  Ñengo Flow, Bad Bunny - Gato de Noche ( Video ...              4
3  Seattle weather conditions: Ongoing power outa...              4
4         Honda’s $225 Million Mistake – Rune Review              4
5                         i became an italian farmer              4
6             What Dan and Phil Text Each Other 2022              4
7  It’s Beginning To Look A Lot Like Christmas (c...              4
8        That '90s Show | Official Trailer | Netflix              4
9          20 Things You Didn't Know About The NBA..              4


## Log transform virality metrics

In [10]:
#the log transformation of view_count to account for varying numbers of metrics
df_2023['log_views'] = np.log(df_2023['view_count'] + 1)

In [11]:
print(df_2023[['view_count', 'log_views']].describe().round(2))

        view_count  log_views
count     12176.00   12176.00
mean    1290155.37      13.35
std     3539901.17       1.00
min           0.00       0.00
25%      308398.25      12.64
50%      545155.50      13.21
75%     1084321.25      13.90
max    91463891.00      18.33


In [12]:
# doing the same for no of likes and comment count
df_2023['log_likes'] = np.log(df_2023['likes'] + 1)
df_2023['log_comments'] = np.log(df_2023['comment_count'] + 1)

In [13]:
print(df_2023[['likes', 'log_likes', 'comment_count', 'log_comments']].describe().round(2))

            likes  log_likes  comment_count  log_comments
count    12176.00   12176.00       12176.00      12176.00
mean     67326.57      10.10        5054.06          7.50
std     220854.46       1.46       21592.82          1.46
min          0.00       0.00           0.00          0.00
25%      11966.75       9.39         937.00          6.84
50%      23988.50      10.09        1840.00          7.52
75%      52986.00      10.88        3777.50          8.24
max    7114337.00      15.78     1086344.00         13.90


## Load category labels

In [14]:
with open('../data/US_category_id.json', 'r') as f:
    category_data = json.load(f)

category_dict = {int(item['id']): item['snippet']['title'] 
                 for item in category_data['items']}

df_2023['category'] = df_2023['categoryId'].map(category_dict)

print(df_2023['category'].value_counts())

category
Gaming                   2550
Entertainment            2343
Music                    1850
Sports                   1773
People & Blogs            947
Film & Animation          500
Comedy                    482
News & Politics           362
Science & Technology      355
Autos & Vehicles          303
Education                 299
Howto & Style             259
Travel & Events            98
Pets & Animals             54
Nonprofits & Activism       1
Name: count, dtype: int64


## Save cleaned data


In [17]:
# save cleaned dataframe to data folder for use in next notebooks
df_2023.to_csv('../data/df_2023_clean.csv', index=False)
print("Saved successfully")
print(f"Final shape: {df_2023.shape}")

Saved successfully
Final shape: (12176, 21)
